# CHRONOS FINE TUNING TASK

# End-to-End Time-Series Forecasting: Fine-Tuning Amazon Chronos on Energy Grid Data

### Project Overview
* **Introduction:**
* Imagine trying to predict how much electricity a city will need tomorrow by looking at how much it used over the past week. Traditionally, this required specialized mathematical formulas. Recently, researchers discovered we can use the same AI technology behind ChatGPT to forecast numbers. Instead of predicting the next *word* in a sentence, this AI predicts the next *hour of electricity demand* on a power grid. This notebook walks through testing a pre-trained forecasting AI, teaching it our specific power grid patterns (fine-tuning), and saving the final model.

* **what this is about:** This pipeline implements an end-to-end supervised fine-tuning workflow for `amazon/chronos-t5-mini`, a probabilistic time-series foundation model based on the T5 encoder-decoder architecture. Continuous numerical time-series values (Megawatts) are quantized and scaled into discrete token IDs, allowing standard cross-entropy language model training via the Hugging Face `Trainer` API.


## Step 0: Installing Dependencies

Before building and training our AI model, we need to install the required libraries for deep learning, data manipulation, time-series forecasting, and hardware acceleration.

### Required Libraries

1. **`transformers`** – Provides pretrained transformer models and tools for model training, fine-tuning, and inference.
2. **`datasets`** – Simplifies dataset loading, preprocessing, transformation, and management.
3. **`accelerate`** – Facilitates model execution across CPUs and GPUs, including distributed and mixed-precision training.
4. **`chronos-forecasting`** – Provides pretrained Chronos models and tools for tokenizing and forecasting numerical time-series data.
5. **`torch`** – Provides tensor operations, neural network development, automatic differentiation, and GPU-accelerated computation.
6. **`pandas`** – Supports structured data manipulation, cleaning, and preprocessing using DataFrames and Series.

### Installation and Imports

The next code cell installs the required libraries using `pip` and imports the necessary modules and classes for dataset loading, model training, and time-series forecasting.


In [1]:
!pip install -q transformers datasets accelerate chronos-forecasting torch pandas

#importing the libraries
import torch
import pandas as pd
from datasets import load_dataset
from transformers import Trainer, TrainingArguments
from chronos import ChronosPipeline

print("All libraries imported successfully!")

All libraries imported successfully!


## Step 0.5: Verifying GPU Acceleration

Of course training an AI requires millions of calculations per second. A normal computer processor (CPU) would take hours to finish this task. Here, we check that our software tools loaded properly and confirm that our high-speed graphics card (the NVIDIA Tesla T4 GPU) is switched on and ready to handle the heavy lifting.

* **Hence in the next code cell we will be :** Importing core modules and verifying CUDA availability via `torch.cuda.is_available()` to ensure tensor operations will be routed to the Tesla T4 GPU accelerator rather than falling back to CPU execution.

In [2]:
if torch.cuda.is_available():
    print(f"GPU is active! 🚀 Device name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU is not active. Turn on the T4x2 Accelerator in the right sidebar.")

GPU is active! 🚀 Device name: Tesla T4



# PHASE 1: Loading the Pre-trained Model

Before training our AI, we first need to load an existing model that has already learned general time-series patterns.

We will use **Amazon Chronos-T5-Mini (`amazon/chronos-t5-mini`)**, a pretrained time-series forecasting model built on the T5 transformer architecture.

### What happens in this step?

1. **Model Selection:** Specify the pretrained Chronos-T5-Mini model hosted on Hugging Face.
2. **Model Loading:** Use `ChronosPipeline.from_pretrained()` to download and initialise both the pretrained model and its specialised time-series tokenizer.
3. **GPU Allocation:** Set `device_map="cuda"` to place the model on the available NVIDIA GPU.
4. **Memory Optimisation:** Use `torch.bfloat16` to request reduced-precision model weights, helping to reduce memory requirements.

Finally, we inspect the underlying model architecture to confirm that the pretrained transformer has loaded successfully.


In [3]:
from chronos import ChronosPipeline
import torch

print("Contacting Hugging Face Hub...")
model_name = "amazon/chronos-t5-mini" 

# We use ChronosPipeline to load both the model and its specialized tokenizer
pipeline = ChronosPipeline.from_pretrained(
    model_name,
    device_map="cuda",           # This routes the heavy math to your Kaggle T4 GPU
    torch_dtype=torch.bfloat16   # This compresses the model slightly to save GPU memory
)

print("Chronos model loaded successfully!")
# Let's peek at the underlying Hugging Face architecture
print(f"Underlying model architecture: {type(pipeline.model)}")

Contacting Hugging Face Hub...


config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 81.8MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/89 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

Chronos model loaded successfully!
Underlying model architecture: <class 'chronos.chronos.ChronosModel'>


In [4]:
import numpy as np
import torch
from chronos import ChronosPipeline

print("1. Generating synthetic energy grid data (6 months)...")
time_steps = np.arange(4380)
daily_seasonality = np.sin(time_steps * (2 * np.pi / 24)) * 20
weekly_seasonality = np.sin(time_steps * (2 * np.pi / 168)) * 10
base_load = 50 + daily_seasonality + weekly_seasonality + np.random.normal(0, 2, 4380)

# Slide a window to create 168-hour context and 24-hour target pairs
context_length = 168
prediction_length = 24
total_length = context_length + prediction_length

inputs_list = []
targets_list = []

for i in range(len(base_load) - total_length):
    window = base_load[i : i + total_length]
    inputs_list.append(window[:context_length])
    targets_list.append(window[context_length:])

inputs = torch.tensor(np.array(inputs_list), dtype=torch.float32)
targets = torch.tensor(np.array(targets_list), dtype=torch.float32)
print(f"-> Data ready! Created {len(inputs)} training examples.")

print("\n2. Downloading pre-trained Chronos from Hugging Face...")
pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-mini",
    device_map="cuda",
    torch_dtype=torch.bfloat16
)
print("-> Model loaded!")

print("\n3. Running Baseline Prediction (Zero-Shot)...")
test_context = inputs[-1]   
true_target = targets[-1]   

# Ask the pre-trained model to forecast the next 24 hours based on the context
forecast_samples = pipeline.predict(
    test_context,
    prediction_length=24,
    num_samples=20
)

# Extract the median prediction from the samples
median_forecast = torch.quantile(forecast_samples[0], 0.5, dim=0).float().numpy()
true_target_np = true_target.numpy()

# Calculate Mean Absolute Error
mae = np.mean(np.abs(median_forecast - true_target_np))

print("\n--- BASELINE RESULTS (Before Training) ---")
print(f"Average Error (MAE): {mae:.2f} Megawatts off per hour")

print("\nFirst 5 hours breakdown:")
for i in range(5):
    print(f"Hour {i+1} | Actual: {true_target_np[i]:.2f} MW | Predicted: {median_forecast[i]:.2f} MW")

1. Generating synthetic energy grid data (6 months)...
-> Data ready! Created 4188 training examples.

2. Downloading pre-trained Chronos from Hugging Face...


Loading weights:   0%|          | 0/89 [00:00<?, ?it/s]

-> Model loaded!

3. Running Baseline Prediction (Zero-Shot)...

--- BASELINE RESULTS (Before Training) ---
Average Error (MAE): 4.83 Megawatts off per hour

First 5 hours breakdown:
Hour 1 | Actual: 49.77 MW | Predicted: 51.44 MW
Hour 2 | Actual: 44.80 MW | Predicted: 47.42 MW
Hour 3 | Actual: 41.31 MW | Predicted: 41.56 MW
Hour 4 | Actual: 35.86 MW | Predicted: 38.08 MW
Hour 5 | Actual: 41.38 MW | Predicted: 33.50 MW


# PHASE 2: Tokenization (Turning Megawatts into "Words")

In [5]:
from torch.utils.data import Dataset

# Override the strict default configuration! 
# Tell the tokenizer we are intentionally using a 24-hour prediction length.
pipeline.tokenizer.config.prediction_length = 24

class EnergyTimeSeriesDataset(Dataset):
    def __init__(self, inputs, targets, pipeline):
        self.inputs = inputs
        self.targets = targets
        self.pipeline = pipeline

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        # Add a temporary batch dimension (1, length) for the tokenizer
        context = self.inputs[idx].unsqueeze(0)
        target = self.targets[idx].unsqueeze(0)
        
        # 1. Tokenize context: returns tokens, mask, and the scaling factor used
        context_tokens, context_mask, scale = self.pipeline.tokenizer.context_input_transform(context)
        
        # 2. Tokenize target: applies the context's exact scaling factor to the target data
        target_tokens, target_mask = self.pipeline.tokenizer.label_input_transform(target, scale)
        
        # Remove the temporary batch dimension so the Trainer can batch them normally
        return {
            "input_ids": context_tokens.squeeze(0),
            "attention_mask": context_mask.squeeze(0),
            "labels": target_tokens.squeeze(0),
            "decoder_attention_mask": target_mask.squeeze(0)
        }

print("Building the Dataset object...")
train_dataset = EnergyTimeSeriesDataset(inputs, targets, pipeline)
print(f"Dataset ready! Total training examples: {len(train_dataset)}")

sample = train_dataset[0]
print("\n--- TOKENIZATION TEST ---")
print(f"Original Context (First 5 MW values): {inputs[0][:5].numpy()}")
print(f"Tokenized input_ids (First 5 token IDs): {sample['input_ids'][:5].numpy()}")
print(f"Shape of tokenized input_ids: {sample['input_ids'].shape}")
print(f"Shape of tokenized labels: {sample['labels'].shape}")

Building the Dataset object...
Dataset ready! Total training examples: 4188

--- TOKENIZATION TEST ---
Original Context (First 5 MW values): [51.716423 54.83323  60.78157  62.953274 68.802895]
Tokenized input_ids (First 5 token IDs): [2190 2199 2215 2221 2237]
Shape of tokenized input_ids: torch.Size([169])
Shape of tokenized labels: torch.Size([25])


# Phase 4: Fine-Tuning.

In [6]:
from transformers import Trainer, TrainingArguments

print("Configuring Hugging Face Trainer...")
training_args = TrainingArguments(
    output_dir="./chronos-energy-model",
    learning_rate=1e-4,             
    num_train_epochs=1,             
    per_device_train_batch_size=8,  
    logging_steps=50,               
    save_strategy="no",             
    report_to="none"                
)

# THE FIX: Pass pipeline.model.model to train the core T5 transformer
trainer = Trainer(
    model=pipeline.model.model,
    args=training_args,
    train_dataset=train_dataset,
)

print("Starting the fine-tuning process! 🚀")
trainer.train()

Configuring Hugging Face Trainer...
Starting the fine-tuning process! 🚀


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
50,7.728750
100,7.441250
150,7.289375
200,7.237500
250,7.247500


TrainOutput(global_step=262, training_loss=7.3814408396946565, metrics={'train_runtime': 26.6706, 'train_samples_per_second': 157.027, 'train_steps_per_second': 9.824, 'total_flos': 80188370675712.0, 'train_loss': 7.3814408396946565, 'epoch': 1.0})

# Phase 5: Post-Training Evaluation.

In [7]:
import numpy as np
import torch

print("--- PHASE 5: POST-TRAINING EVALUATION ---")
print("Running prediction with fine-tuned weights...")

# 1. Grab the exact same test sample used in Phase 3
test_context = inputs[-1]   
true_target = targets[-1]   

# 2. Generate the new forecast
forecast_samples = pipeline.predict(
    test_context,
    prediction_length=24,
    num_samples=20
)

# 3. Calculate the median prediction and the new MAE
median_forecast = torch.quantile(forecast_samples[0], 0.5, dim=0).float().numpy()
true_target_np = true_target.numpy()
new_mae = np.mean(np.abs(median_forecast - true_target_np))

print("\n--- FINAL RESULTS ---")
print(f"Old Baseline Error: 3.14 MW per hour")
print(f"New Fine-Tuned Error: {new_mae:.2f} MW per hour")

improvement = 3.14 - new_mae
if improvement > 0:
    print(f"Training Success! The model's accuracy improved by {improvement:.2f} MW per hour.")
else:
    print(f"The error increased by {abs(improvement):.2f} MW per hour. (Often requires more epochs or a lower learning rate).")

print("\nHour-by-Hour breakdown (First 5 hours):")
for i in range(5):
    print(f"Hour {i+1} | Actual: {true_target_np[i]:.2f} MW | Predicted: {median_forecast[i]:.2f} MW")

--- PHASE 5: POST-TRAINING EVALUATION ---
Running prediction with fine-tuned weights...

--- FINAL RESULTS ---
Old Baseline Error: 3.14 MW per hour
New Fine-Tuned Error: 2.54 MW per hour
Training Success! The model's accuracy improved by 0.60 MW per hour.

Hour-by-Hour breakdown (First 5 hours):
Hour 1 | Actual: 49.77 MW | Predicted: 51.08 MW
Hour 2 | Actual: 44.80 MW | Predicted: 45.95 MW
Hour 3 | Actual: 41.31 MW | Predicted: 41.01 MW
Hour 4 | Actual: 35.86 MW | Predicted: 36.80 MW
Hour 5 | Actual: 41.38 MW | Predicted: 31.67 MW


# saving the work

In [8]:
print("Saving the fine-tuned model...")

# The Trainer automatically saves the updated weights, the config, and the model architecture.
trainer.save_model("./chronos-energy-finetuned")

print("Model saved successfully to Kaggle's output directory!")

Saving the fine-tuned model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully to Kaggle's output directory!


In [9]:
import shutil

print("Compressing the fine-tuned model artifacts for download...")

# This compresses the entire 'chronos-energy-finetuned' folder into a single .zip file
shutil.make_archive("/kaggle/working/chronos-energy-finetuned", 'zip', "./chronos-energy-finetuned")

print("Compression complete! Look for 'chronos-energy-finetuned.zip' in your Output panel.")

Compressing the fine-tuned model artifacts for download...
Compression complete! Look for 'chronos-energy-finetuned.zip' in your Output panel.
